In [ ]:
# Databricks notebook source
# COMMAND ----------

# Tag all Spark SQL queries for tracking
spark.conf.set('spark.databricks.queryTag.data_origin', 'open_data_portal')

dbutils.widgets.text("catalog_name", "", "Catalog Name")
dbutils.widgets.text("meta_schema", "meta", "Control Schema Name")
dbutils.widgets.text("job_start_time", "", "Job Start Time (ISO)")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
META_SCHEMA = dbutils.widgets.get("meta_schema")
JOB_START_TIME = dbutils.widgets.get("job_start_time").strip()

QUEUE_TABLE = f"`{CATALOG_NAME}`.`{META_SCHEMA}`.`_socrata_ingestion_queue`"

# Default to looking back 2 hours if job_start_time is not passed
time_filter = (
    f"logged_at >= TIMESTAMP '{JOB_START_TIME}'"
    if JOB_START_TIME
    else "logged_at >= current_timestamp() - INTERVAL 4 HOURS"
)

failures_df = spark.sql(f"""
    SELECT
        dataset_id,
        task_type,
        table_name,
        status,
        left(error_message, 300) AS error_preview,
        logged_at
    FROM {QUEUE_TABLE}
    WHERE {time_filter}
      AND status IN ('FAILED', 'QUARANTINED')
    ORDER BY logged_at DESC
""")

failures = failures_df.collect()

if failures:
    summary_lines = [
        f"- [{r['task_type']}] {r['dataset_id']} ({r['table_name']}) -> {r['status']}: {r['error_preview']}"
        for r in failures
    ]
    report = "\n".join(summary_lines)
    raise RuntimeError(f"Pipeline completed with {len(failures)} dataset failure(s):\n{report}")

dbutils.notebook.exit("ALL_CLEAR: 0 failures logged.")